In [22]:
# # Viewing timeframe jumps

# def show_jumps(grouped):
#     frame_pattern = re.compile(r'_frame_(\d+)_')

#     inconsistent = {}
#     step_counts = defaultdict(int)

#     for key, files in grouped.items():
#         frame_nums = sorted(int(frame_pattern.search(f).group(1)) for f in files)
#         steps = set(frame_nums[i+1] - frame_nums[i] for i in range(len(frame_nums) - 1))
        
#         if len(steps) > 1:
#             inconsistent[key] = steps
#         elif steps:
#             step_counts[next(iter(steps))] += 1

#     print("Step size distribution across all videos:")
#     for step, count in sorted(step_counts.items()):
#         print(f"  step={step}: {count} video(s)")

#     if inconsistent:
#         print(f"\nVideos with inconsistent frame steps ({len(inconsistent)}):")
#         for key, steps in inconsistent.items():
#             print(f"  {key}: {steps}")
#     else:
#         print("\nAll videos have consistent frame steps.")

Dataset information
FF_data:
DFD -> Google's deepfake dection dataset
fake and real are faceforensics data

In CSV: 
0 - authentic videos
1 - deepfakes

In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None

In [4]:
# Download and process data into frames

# Download all data
# !bash FaceForensics/download_ff_data.sh

# Download 10 videos data
!bash download_ff_data.sh --num_videos 10

#Download one video per category
# !bash download_ff_data.sh --sample

  FaceForensics++ Downloader + Extractor
  Mode:        PARTIAL (1 videos each)
  Compression: c40
  Server:      EU2
  Output:      /Users/cvancaemelbecke/Documents/INF-Deep_Learning/FF_data

  STEP 1: Downloading videos...

----------------------------------------------
Downloading: Original (YouTube real videos)
  Dataset: original
  Output:  /Users/cvancaemelbecke/Documents/INF-Deep_Learning/FF_data/real
----------------------------------------------
By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://kaldir.vc.in.tum.de/faceforensics/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.
Output path: /Users/cvancaemelbecke/Documents/INF-Deep_Learning/FF_data/real/original_sequences/youtube/c40/videos
100%|█████████████████████████████████████████████| 1/1 [00:01<00:00,  1.90s/it]
✓ Done: Original (YouTube real videos)

----------------------------------------------
Downloading: Face2Fac

In [2]:
'''
group videos by name to get into format of map
'''

from collections import defaultdict
import re
import os
import pandas as pd

def group_by_video(directory):
    groups = defaultdict(list)
    for video_name in os.listdir(directory):
        video_path = os.path.join(directory, video_name)
        if not os.path.isdir(video_path):
            continue
        frames = sorted(f for f in os.listdir(video_path) if f.endswith('.png'))
        if frames:
            groups[video_name] = [os.path.join(video_name, f) for f in frames]
    
    result = []
    for key, frames in groups.items():
        result.append([key, frames])
    return result


#print(group_by_video("FF_data/real/original_sequences/youtube/c40/images"))

In [ ]:
'''
create csv storing:
Video_name, frame 1, frame 2, .. frame 10
'''

all_dirs = ['FF_data/real/original_sequences/youtube/c40/images', 
            'FF_data/fake/manipulated_sequences/Deepfakes/c40/images',
            'FF_data/DFD_real/original_sequences/actors/c40/images',
            'FF_data/DFD_fake/manipulated_sequences/DeepFakeDetection/c40/images',
            'FF_data/Face2Face/manipulated_sequences/Face2Face/c40/images',
            'FF_data/FaceSwap/manipulated_sequences/FaceSwap/c40/images',
            'FF_data/NeuralTextures/manipulated_sequences/NeuralTextures/c40/images'
            ]

'''
0 - Authentic
1 - Deepfake
'''

def create_datamap(all_dirs):
    data = {
        'root_dir': [],
        'video_name': []
    }
    for i in range(10):
        data['frame_' + str(i)] = []
    
    data['label'] = []

    for dir in all_dirs:
        label = 1
        if 'original' in dir:
            label = 0
        video_map = group_by_video(dir)
        for video, frames in video_map:
            i = 0
            for j in range(10, len(frames), 10):
                split = frames[i:j]
                i = j
                data['root_dir'].append(dir)
                data['video_name'].append(video)
                data['label'].append(label)
                for k in range(len(split)):
                    data['frame_' + str(k)].append(split[k])
    return data

data_map = create_datamap(all_dirs)

df = pd.DataFrame(data_map)
#df.to_csv('dataset_splits.csv', index=False) # for testing dataset


In [77]:
from torchvision import transforms

class FrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Returns a clip tensor of shape (3, T, H, W) = (3, 10, 500, 500).
        The DataLoader will stack these into (B, 3, T, H, W).
        """
        root_dir = self.df.iloc[idx, 0]
        tensors = []
        for i in range(10):
            fname = self.df.iloc[idx, i + 2]  # cols 2–11 are frame_0–frame_9
            path = os.path.join(root_dir, fname)
            img = Image.open(path).convert('RGB').resize((500, 500))
            tensors.append(self.transform(img)) # (3, 500, 500)
        clip = torch.stack(tensors, dim=1) # (3, T, 500, 500)
        label = self.df.iloc[idx, -1]
        return clip, label


dataset = FrameDataset(df)
test_loader = DataLoader(dataset)
for frames, label in test_loader:
    print(frames.shape)
    print(label)
    break


torch.Size([1, 3, 10, 500, 500])
tensor([0])
